# Classical Time Series Forecasting

1. **Time Series Decomposition** - Trend, seasonality, residual
2. **Stationarity** - ADF test, differencing
3. **ARIMA** - AutoRegressive Integrated Moving Average
4. **Seasonal ARIMA (SARIMA)** - Handling seasonality
5. **Prophet** - Facebook's automated forecasting

**Dataset**: Airline Passengers (classic time series dataset)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")

In [ ]:
# Load airline passengers dataset
# Classic dataset: monthly totals of international airline passengers, 1949-1960
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=["Month"], index_col="Month")
df.columns = ["passengers"]
df.index.freq = "MS"  # Month start

print(f"Shape: {df.shape}, Period: {df.index[0]} to {df.index[-1]}")
df.plot(figsize=(12, 4), title="International Airline Passengers", color="teal")
plt.ylabel("Passengers (thousands)")
plt.tight_layout()
plt.show()

## 1. Time Series Decomposition

In [ ]:
decomposition = seasonal_decompose(df["passengers"], model="multiplicative", period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
decomposition.observed.plot(ax=axes[0], title="Observed", color="teal")
decomposition.trend.plot(ax=axes[1], title="Trend", color="coral")
decomposition.seasonal.plot(ax=axes[2], title="Seasonal", color="steelblue")
decomposition.resid.plot(ax=axes[3], title="Residual", color="gray")
plt.tight_layout()
plt.show()

## 2. Stationarity Testing

In [ ]:
def test_stationarity(series, name=""):
    result = adfuller(series.dropna())
    is_stationary = result[1] < 0.05
    print(f"{name} ADF Statistic: {result[0]:.4f}, p-value: {result[1]:.4f} -> {'Stationary' if is_stationary else 'Non-stationary'}")
    return is_stationary

test_stationarity(df["passengers"], "Original")
test_stationarity(df["passengers"].diff().dropna(), "1st Difference")
test_stationarity(df["passengers"].diff().diff(12).dropna(), "1st + Seasonal Diff")

In [ ]:
# ACF and PACF plots help determine ARIMA orders
diff_series = df["passengers"].diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(diff_series, ax=axes[0], lags=24, title="ACF (1st Difference)")
plot_pacf(diff_series, ax=axes[1], lags=24, title="PACF (1st Difference)")
plt.tight_layout()
plt.show()

## 3. SARIMA Model

SARIMA(p,d,q)(P,D,Q,s):
- (p,d,q) = non-seasonal (AR order, differencing, MA order)
- (P,D,Q,s) = seasonal components with period s

In [ ]:
# Train/test split (last 24 months as test)
train = df["passengers"][:-24]
test = df["passengers"][-24:]

# Fit SARIMA
model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
results = model.fit(disp=False)
print(results.summary().tables[1])

In [ ]:
# Forecast
forecast = results.forecast(steps=24)
forecast_ci = results.get_forecast(steps=24).conf_int()

plt.figure(figsize=(12, 5))
plt.plot(train.index, train, label="Training", color="teal")
plt.plot(test.index, test, label="Actual", color="black", linewidth=2)
plt.plot(test.index, forecast, label="SARIMA Forecast", color="coral", linestyle="--")
plt.fill_between(test.index, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1], alpha=0.2, color="coral")
plt.title("SARIMA Forecast vs Actual")
plt.legend()
plt.tight_layout()
plt.show()

mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test - forecast) / test)) * 100
print(f"MAE: {mae:.1f}, RMSE: {rmse:.1f}, MAPE: {mape:.1f}%")

## 4. Prophet (Optional)

Facebook Prophet is designed for business time series with strong seasonal effects.

In [ ]:
# Uncomment to run (requires: pip install prophet)
# from prophet import Prophet
#
# # Prophet requires columns named 'ds' and 'y'
# prophet_df = df.reset_index()
# prophet_df.columns = ["ds", "y"]
# prophet_train = prophet_df[:-24]
# prophet_test = prophet_df[-24:]
#
# m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
# m.fit(prophet_train)
#
# future = m.make_future_dataframe(periods=24, freq="MS")
# forecast = m.predict(future)
#
# fig = m.plot(forecast)
# plt.title("Prophet Forecast")
# plt.show()
#
# fig2 = m.plot_components(forecast)
# plt.show()

## Key Takeaways

1. **Decompose first** - understand trend, seasonality, and residuals before modeling
2. **Stationarity is key for ARIMA** - use differencing and the ADF test
3. **ACF/PACF guide parameter selection** - or use `auto_arima` from pmdarima
4. **SARIMA handles seasonality** natively - specify the seasonal period
5. **Prophet is great for business series** - handles holidays, changepoints, and missing data automatically
6. **Always use temporal train/test splits** - never shuffle time series data